# Exercise
Download heart disease dataset heart.csv in Exercise folder and do following, (credits of dataset: https://www.kaggle.com/fedesoriano/heart-failure-prediction)

1. Load heart disease dataset in pandas dataframe
2. Remove outliers using Z score. Usual guideline is to remove anything that has Z score > 3 formula or Z score < -3
3. Convert text columns to numbers using label encoding and one hot encoding
4. Apply scaling
5. Build a classification model using various methods (SVM, logistic regression, random forest) and check which model gives you the best accuracy
6. Now use PCA to reduce dimensions, retrain your model and see what impact it has on your model in terms of accuracy. Keep in mind that many times doing PCA reduces the accuracy but computation is much lighter and that's the trade off you need to consider while building models in real life

In [286]:
import pandas as pd
df = pd.read_csv("heart.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [287]:
df.shape

(918, 12)

In [288]:
df.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


# Treat Outlier

In [289]:
# Resting BP

df1 = df[df.RestingBP <= (3*df.RestingBP.std() + df.RestingBP.mean())]


In [290]:
df1.shape

(911, 12)

In [291]:
# Cholesterol

df2 = df1[df1.Cholesterol <= (3*df1.Cholesterol.std() + df1.Cholesterol.mean())]
df2.shape

(908, 12)

In [292]:
# FastingBS	
df3 = df2[df2.FastingBS <= (3*df2.FastingBS.std() + df2.FastingBS.mean())]
df3.shape

(908, 12)

In [293]:
# MaxHR
df4 = df3[df3.MaxHR <= (3*df3.MaxHR.std() + df3.MaxHR.mean())]
df4.shape

(908, 12)

In [294]:
# Oldpeak
df5 = df4[df4.Oldpeak <=(3*df4.Oldpeak.std() + df4.Oldpeak.mean())]
df5.shape

(902, 12)

In [295]:
# HeartDisease
df6 = df5[df5.HeartDisease <= (3*df5.HeartDisease.std() + df5.HeartDisease.mean())]
df6.shape

(902, 12)

In [296]:
df6.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [297]:
df6.nunique()

Age                50
Sex                 2
ChestPainType       4
RestingBP          64
Cholesterol       219
FastingBS           2
RestingECG          3
MaxHR             117
ExerciseAngina      2
Oldpeak            48
ST_Slope            3
HeartDisease        2
dtype: int64

In [298]:
df6.info()

<class 'pandas.DataFrame'>
Index: 902 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             902 non-null    int64  
 1   Sex             902 non-null    str    
 2   ChestPainType   902 non-null    str    
 3   RestingBP       902 non-null    int64  
 4   Cholesterol     902 non-null    int64  
 5   FastingBS       902 non-null    int64  
 6   RestingECG      902 non-null    str    
 7   MaxHR           902 non-null    int64  
 8   ExerciseAngina  902 non-null    str    
 9   Oldpeak         902 non-null    float64
 10  ST_Slope        902 non-null    str    
 11  HeartDisease    902 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 91.6 KB


In [299]:
df7 = df6.copy()

df7 = df7.replace({
    "ExerciseAngina": {
        "N": 0,
        "Y": 1
    },
    "ST_Slope": {
        "Down": 1,
        "Flat": 2,
        "Up": 3
    },
    "RestingECG": {
        "Normal": 1,
        "ST": 2,
        "LVH": 3
    }
})

df7.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,1,172,0,0.0,3,0
1,49,F,NAP,160,180,0,1,156,0,1.0,2,1
2,37,M,ATA,130,283,0,2,98,0,0.0,3,0
3,48,F,ASY,138,214,0,1,108,1,1.5,2,1
4,54,M,NAP,150,195,0,1,122,0,0.0,3,0


In [300]:
df8 = pd.get_dummies(df7, drop_first=True, dtype = int)
df8.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_2,RestingECG_3,ExerciseAngina_1,ST_Slope_2,ST_Slope_3
0,40,140,289,0,172,0.0,0,1,1,0,0,0,0,0,0,1
1,49,160,180,0,156,1.0,1,0,0,1,0,0,0,0,1,0
2,37,130,283,0,98,0.0,0,1,1,0,0,1,0,0,0,1
3,48,138,214,0,108,1.5,1,0,0,0,0,0,0,1,1,0
4,54,150,195,0,122,0.0,0,1,0,1,0,0,0,0,0,1


In [301]:
X = df8.drop("HeartDisease",axis='columns')
y = df8.HeartDisease

X.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_2,RestingECG_3,ExerciseAngina_1,ST_Slope_2,ST_Slope_3
0,40,140,289,0,172,0.0,1,1,0,0,0,0,0,0,1
1,49,160,180,0,156,1.0,0,0,1,0,0,0,0,1,0
2,37,130,283,0,98,0.0,1,1,0,0,1,0,0,0,1
3,48,138,214,0,108,1.5,0,0,0,0,0,0,1,1,0
4,54,150,195,0,122,0.0,1,0,1,0,0,0,0,0,1


In [302]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled

array([[-1.42896269,  0.46089071,  0.85238015, ..., -0.82065181,
        -1.00221976,  1.13805334],
       [-0.47545956,  1.5925728 , -0.16132855, ..., -0.82065181,
         0.99778516, -0.87869344],
       [-1.74679706, -0.10495034,  0.79657967, ..., -0.82065181,
        -1.00221976,  1.13805334],
       ...,
       [ 0.37209878, -0.10495034, -0.61703246, ...,  1.21854359,
         0.99778516, -0.87869344],
       [ 0.37209878, -0.10495034,  0.35947592, ..., -0.82065181,
         0.99778516, -0.87869344],
       [-1.64085227,  0.3477225 , -0.20782894, ..., -0.82065181,
        -1.00221976,  1.13805334]], shape=(902, 15))

In [303]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size= 0.2,random_state = 30)

In [304]:
X_train.shape

(721, 15)

In [305]:
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier()
model_rf.fit(X_train, y_train)
model_rf.score(X_test, y_test)

0.8839779005524862

## Use PCA

In [306]:
from sklearn.decomposition import PCA

pca = PCA(0.95)
X_pca = pca.fit_transform(X)
X_pca

array([[ 93.82508417,  29.40267731],
       [-15.58386089,  14.10477683],
       [ 83.29518824, -38.68514771],
       ...,
       [-67.57283048, -17.61633189],
       [ 40.70396358,  33.38465901],
       [-19.91326521,  37.29256128]], shape=(902, 2))

In [307]:
X_train_pca, X_test_pca, y_train, y_test = train_test_split(X_pca, y, test_size = 0.2, random_state=30)

In [308]:
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier()
model_rf.fit(X_train_pca, y_train)
model_rf.score(X_test_pca, y_test)

0.7237569060773481